# Total-Site Targets and SUGCC
This notebook shows when a site-level view changes the utility story enough that process-only targeting is no longer the right decision frame.


## Case context
The packaged `zonal_site.json` case is a multi-utility site with several subzones. The workflow is: compare direct, total-process, and total-site targets; inspect the graphs that matter at each level; then run the cogeneration screen to see whether the chosen utility picture offers any above-pinch power opportunity.


In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case("zonal_site.json", work_dir / "zonal_site.json")
problem = PinchProblem(problem_filepath=case_path)
summary = problem.summary_frame()
summary.loc[
    summary["Target"].isin(
        [
            "zonal_site/Direct Integration",
            "zonal_site/Total Process Target",
            "zonal_site/Total Site Target",
        ]
    ),
    ["Target", "Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
].reset_index(drop=True)


,Target,Hot Utility Target,Cold Utility Target,Heat Recovery
0,zonal_site/Direct Integration,0.000000,172.680000,53.142000
1,zonal_site/Total Process Target,27.078235,199.758235,26.063765
2,zonal_site/Total Site Target,0.000000,172.680000,53.142000


In [5]:
graph_catalog = problem.graph_catalog()
key_graphs = graph_catalog.loc[
    graph_catalog["Graph Type"].isin(
        [
            "Grand Composite Curve",
            "Total Site Profiles",
            "Site Utility Grand Composite Curve",
        ]
    )
].reset_index(drop=True)

root_gcc = problem.plot_grand_composite_curve()
total_site_profiles = problem.plot(graph_type="Total Site Profiles")
sugcc = problem.plot(graph_type="Site Utility Grand Composite Curve")
{
    "catalog_rows": key_graphs.to_dict(orient="records"),
    "root_gcc_traces": len(root_gcc.data),
    "total_site_profile_traces": len(total_site_profiles.data),
    "sugcc_traces": len(sugcc.data),
}


{'catalog_rows': [{'Zone': 'Direct Integration',
   'Graph Type': 'Grand Composite Curve',
   'Graph Name': 'Graph 4',
   'Index': 3},
  {'Zone': 'Total Site Target',
   'Graph Type': 'Total Site Profiles',
   'Graph Name': 'Graph 1',
   'Index': 0},
  {'Zone': 'Total Site Target',
   'Graph Type': 'Site Utility Grand Composite Curve',
   'Graph Name': 'Graph 2',
   'Index': 1}],
 'root_gcc_traces': 4,
 'total_site_profile_traces': 4,
 'sugcc_traces': 7}

In [6]:
cogeneration_problem = PinchProblem(problem_filepath=case_path)
cogeneration_problem.target()
cogeneration_target = cogeneration_problem.target_cogeneration(
    options={
        "DO_TURBINE_WORK": True,
        "base_target_type": "Total Site Target",
    }
)
{
    "target": cogeneration_target.name,
    "work_target": cogeneration_target.work_target,
    "turbine_efficiency_target": cogeneration_target.turbine_efficiency_target,
}


{'target': 'zonal_site/Direct Integration',
 'work_target': 0.0,
 'turbine_efficiency_target': 0.0}

## Inspect these outputs first
Use the three-row summary table to decide whether the site-level view changes the utility picture materially. Then move to the total-site profiles and the Site Utility Grand Composite Curve. Those plots answer different questions than the process GCC: they show utility-system interaction rather than only process-source and process-sink overlap.

## Interpretation
If the total-site targets differ meaningfully from the direct-integration row, switch to a site utility planning mindset before discussing retrofit actions. The cogeneration screen is a follow-on check: a zero or near-zero work target means the current site utility structure is not exposing an above-pinch steam-power opportunity under the chosen assumptions.
